# Lưu UPT hidden-pair cache thành Kaggle Dataset bằng API token

CPU, Internet **ON**. Tự tải đúng **Output version 56** của
`ducanh2403/2-vi-cai-kia-qua-nhieu` bằng API token; không cần gắn Input.
Không dùng version mới nhất (57), không chạy lại notebook nguồn.

1. Lấy token mới tại https://www.kaggle.com/settings/api (không dùng nội dung kaggle.json).
2. Add-ons → Secrets: thêm và bật secret tên **KAGGLE_API_TOKEN**.
3. Sửa OWNER đúng tài khoản sở hữu token. Chạy các cell theo thứ tự.

Chỉ tạo dataset **Private mới**, không sửa/xóa dataset đã có. Nếu slug đã tồn tại,
mở dataset kiểm tra trước; đổi SLUG khi thực sự muốn tạo bản khác.
Token chỉ nằm trong bộ nhớ và môi trường tiến trình con; không ghi vào Output.
Không chạy UPT, caption model hay training.


In [ ]:
from pathlib import Path
KERNEL_VERSION = 'ducanh2403/2-vi-cai-kia-qua-nhieu/56'
OWNER = 'ducanh2403'  # Đổi nếu token thuộc tài khoản khác
SLUG = 'upt-hidden-pairs-caption11k'
TITLE = 'UPT hidden pairs caption 11k'
DESTINATION = Path('/kaggle/working/upt_hidden_pairs_api_dataset')


In [ ]:
import os
import re
import subprocess
import sys
from kaggle_secrets import UserSecretsClient

if not re.fullmatch(r'[a-zA-Z0-9_-]+', OWNER) or not re.fullmatch(r'[a-z0-9]+(?:-[a-z0-9]+)*', SLUG):
    raise ValueError('OWNER/SLUG không hợp lệ.')
if not 3 <= len(SLUG) <= 50 or not 6 <= len(TITLE) <= 50:
    raise ValueError('SLUG cần 3–50 ký tự, TITLE cần 6–50 ký tự.')
try:
    token = UserSecretsClient().get_secret('KAGGLE_API_TOKEN').strip()
    if not token:
        raise ValueError('empty')
except Exception:
    raise RuntimeError('Thêm và bật secret KAGGLE_API_TOKEN tại Add-ons > Secrets. Lấy token tại https://www.kaggle.com/settings/api; bật Internet.') from None

def run_cli(args, authenticated=True):
    environment = os.environ.copy()
    for key in ('KAGGLE_API_TOKEN', 'KAGGLE_USERNAME', 'KAGGLE_KEY'):
        environment.pop(key, None)
    if authenticated:
        environment['KAGGLE_API_TOKEN'] = token
    environment['PYTHONUNBUFFERED'] = '1'
    process = subprocess.Popen(args, env=environment, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        print(line.replace(token, '[REDACTED]'), end='', flush=True)
    code = process.wait()
    if code:
        raise RuntimeError(f'Lệnh thất bại (exit {code}); xem log đã ẩn token phía trên. Kiểm tra Internet, token, OWNER và slug đã tồn tại.')

# Install before importing kaggle; no credential files are created.
run_cli([sys.executable, '-m', 'pip', 'install', '--upgrade', '--quiet', 'kaggle'], authenticated=False)
print('Secret loaded; CLI ready. Token is not displayed.', flush=True)


In [ ]:
import tempfile

if KERNEL_VERSION != 'ducanh2403/2-vi-cai-kia-qua-nhieu/56':
    raise ValueError('Notebook này khóa Output version 56; không tự chuyển sang latest.')
# Fresh staging prevents stale files from another version being reused.
DOWNLOAD = Path(tempfile.mkdtemp(prefix='upt-hidden-v56-', dir='/tmp'))
print('Downloading only six cache payload files from:', KERNEL_VERSION, flush=True)
run_cli([sys.executable, '-m', 'kaggle', 'kernels', 'output', KERNEL_VERSION,
         '-p', str(DOWNLOAD), '--file-pattern',
         r'(^|/)(cache_receipt\.json|cohort\.json|features_rank[01]\.h5|pairs_rank[01]\.jsonl)$'])
receipts = list(DOWNLOAD.rglob('cache_receipt.json'))
if len(receipts) != 1:
    raise RuntimeError(f'Version 56 trả về {len(receipts)} cache receipt; dừng, không lấy version 57 thay thế.')
SOURCE = receipts[0].parent
print('Downloaded version 56. Next: verify receipt and every payload hash.', flush=True)


In [ ]:
import hashlib
import json
SOURCE_CODE = '"""Copy a verified hidden-pair cache to a small Kaggle Dataset-ready folder; stdlib only."""\nimport hashlib\nimport json\nfrom pathlib import Path\nimport shutil\n\nRECEIPT_SHA = \'873f6da0e1b4c4f7d7408b67c593c07ce56e5b4ca3fa1db5f9ce5e04aa188920\'\nREQUIRED = {\'cohort.json\', \'features_rank0.h5\', \'features_rank1.h5\', \'pairs_rank0.jsonl\', \'pairs_rank1.jsonl\'}\n\n\ndef sha(path):\n    result = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for block in iter(lambda: stream.read(8 * 1024**2), b\'\'):\n            result.update(block)\n    return result.hexdigest()\n\n\ndef locate(source=None):\n    if source is not None:\n        folder = Path(source)\n        if not (folder / \'cache_receipt.json\').is_file():\n            raise FileNotFoundError(f\'SOURCE must contain cache_receipt.json: {folder}\')\n        return folder\n    local = Path(\'/kaggle/working/upt_hidden_pairs_11k\')\n    if (local / \'cache_receipt.json\').is_file() and sha(local / \'cache_receipt.json\') == RECEIPT_SHA:\n        return local\n    candidates = [p.parent for p in Path(\'/kaggle/input\').rglob(\'cache_receipt.json\') if sha(p) == RECEIPT_SHA]\n    if len(candidates) != 1:\n        raise RuntimeError(f\'Found {len(candidates)} matching cache folders. Attach completed cache notebook Output, then set SOURCE to its exact folder. The review ZIP alone is insufficient.\')\n    return candidates[0]\n\n\ndef package(source, destination, receipt_sha=RECEIPT_SHA):\n    source, destination = Path(source).resolve(), Path(destination).resolve()\n    if source == destination or source in destination.parents or destination in source.parents:\n        raise ValueError(\'Source and destination must be separate, non-nested folders.\')\n    receipt_path = source / \'cache_receipt.json\'\n    if sha(receipt_path) != receipt_sha:\n        raise ValueError(\'Receipt differs from the audited cache; do not silently repackage another run.\')\n    receipt = json.loads(receipt_path.read_text(encoding=\'utf-8\'))\n    if receipt[\'status\'] != \'CACHE_COMPLETE\' or receipt[\'schema\'] != \'upt_hidden_pair_caption_cache_v1\':\n        raise ValueError(\'Incomplete or unknown cache schema.\')\n    if set(receipt[\'files\']) != REQUIRED:\n        raise ValueError(\'Cache payload inventory differs.\')\n    expected = dict(receipt[\'files\'], **{\'cache_receipt.json\': receipt_sha})\n    for name, expected_sha in expected.items():\n        path = source / name\n        if not path.is_file() or sha(path) != expected_sha:\n            raise ValueError(\'Missing or changed source file: \' + name)\n        print(\'Source verified:\', name, flush=True)\n    destination.mkdir(parents=True, exist_ok=True)\n    if any(p.name not in set(expected) | {\'transport_receipt.json\'} | {n + \'.partial\' for n in expected}\n           for p in destination.iterdir()):\n        raise ValueError(\'Destination contains unrelated files; choose a fresh folder.\')\n    needed = sum((source / name).stat().st_size for name in expected if not (destination / name).is_file())\n    if shutil.disk_usage(destination).free < needed + 256 * 1024**2:\n        raise RuntimeError(\'Not enough free disk space for cache copy plus256MiB.\')\n    for name, expected_sha in expected.items():\n        final = destination / name\n        if final.exists():\n            if not final.is_file() or sha(final) != expected_sha:\n                raise ValueError(\'Existing destination differs; preserving it: \' + name)\n            print(\'Already verified:\', name, flush=True)\n            continue\n        temporary = destination / (name + \'.partial\')\n        with (source / name).open(\'rb\') as reader, temporary.open(\'wb\') as writer:\n            shutil.copyfileobj(reader, writer, 8 * 1024**2)\n        if sha(temporary) != expected_sha:\n            raise ValueError(\'Copied bytes differ: \' + name)\n        temporary.replace(final)\n        print(\'Copied and verified:\', name, flush=True)\n    report = dict(status=\'DATASET_FOLDER_READY\', source=str(source), payload_files=6,\n                  files=expected, bytes=sum((destination / n).stat().st_size for n in expected),\n                  model_or_training_executed=False,\n                  next=\'Save notebook Output, then create a private Kaggle Dataset from this output folder. No API token required.\')\n    (destination / \'transport_receipt.json\').write_text(json.dumps(report, indent=2) + \'\\n\', encoding=\'utf-8\')\n    print(\'DATASET FOLDER READY:\', destination, flush=True)\n    print(\'Payload MiB:\', round(report[\'bytes\'] / 1024**2, 2), flush=True)\n    return report\n'
assert hashlib.sha256(SOURCE_CODE.encode()).hexdigest() == 'f40458c5c996fa88156c5cce0c89a8d0102b3ef181ce0ad32cf06cbed3ef57c8'
exec(compile(SOURCE_CODE, "package_cache_dataset.py", "exec"))
source = locate(SOURCE)
report = package(source, DESTINATION)
report['source_kernel_version'] = KERNEL_VERSION
report['next'] = 'Upload verified flat payload to a private Kaggle Dataset.'
(DESTINATION / 'transport_receipt.json').write_text(json.dumps(report, indent=2) + '\n', encoding='utf-8')
print('Verified cache ready for API upload.', flush=True)


In [ ]:
metadata_path = DESTINATION / 'dataset-metadata.json'
metadata = {
    'title': TITLE,
    'id': f'{OWNER}/{SLUG}',
    'licenses': [{'name': 'copyright-authors'}],
    'description': 'Private research cache: frozen UPT hidden interaction features for caption train10000/dev1000. No original images. Original data/model rights retained; see cache receipt for provenance. Not a caption quality result.',
}
if metadata_path.exists():
    raise RuntimeError('Có metadata cũ trong staging; dùng DESTINATION mới để tránh ghi đè.')
# Keep raw HDF5/JSONL unchanged; only the flat verified payload is uploaded.
try:
    metadata_path.write_text(json.dumps(metadata, indent=2), encoding='utf-8')
    run_cli([sys.executable, '-m', 'kaggle', 'datasets', 'create',
             '-p', str(DESTINATION), '--keep-tabular', '--dir-mode', 'skip'])
finally:
    metadata_path.unlink(missing_ok=True)

print('Upload request accepted. Kaggle may still be processing the dataset.', flush=True)
print(f'https://www.kaggle.com/datasets/{OWNER}/{SLUG}', flush=True)
run_cli([sys.executable, '-m', 'kaggle', 'datasets', 'status', f'{OWNER}/{SLUG}'])


## Sau upload

Mở link dataset và đợi xử lý hoàn tất, kiểm tra đủ `features_rank0.h5`,
`features_rank1.h5`, `pairs_rank0.jsonl`, `pairs_rank1.jsonl`, `cohort.json`,
`cache_receipt.json` (có thêm transport receipt). Gắn dataset này vào A và H.

Cache nằm **ngay gốc dataset**, vì vậy sửa `HIDDEN_CACHE` theo đường dẫn thực tế, ví dụ:
`/kaggle/input/datasets/ducanh2403/upt-hidden-pairs-caption11k`
(không thêm `/upt_hidden_pairs_11k`). A/H vẫn kiểm tra hash trước training.

Upload được API nhận chưa đồng nghĩa dataset đã xử lý xong. Nếu status chưa ready,
kiểm tra trên trang dataset; không chạy lại cell create liên tục.

Tài liệu chính thức: [token](https://github.com/Kaggle/kaggle-cli/blob/main/docs/README.md#authentication),
[dataset create/status](https://github.com/Kaggle/kaggle-cli/blob/main/docs/datasets.md).
